In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
filter_assignments_onlyMat.py

目的:
- features_OH_onlyMat.csv / features_FP_onlyMat.csv に記載の“材料由来のみ”の変数名と一致する行だけ、
  変数クラスタリング結果 (ClusterAssign_..._OH_*.csv / _FP_*.csv) から抽出して再保存する。
- top3 / cumeig × 各指標 × OH/FP の全ファイルを一括処理。

出力:
- .../OH/04_assignments_onlyMat/ClusterAssign_{...}_OH_*_onlyMat.csv
- .../FP/04_assignments_onlyMat/ClusterAssign_{...}_FP_*_onlyMat.csv
- それぞれに index CSV (処理ログ) を出力

使い方例:
python filter_assignments_onlyMat.py \
  --base /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No \
  --run-root 20251026_under_25clusters/STEP3.3_U25_20251027_correct \
  --oh-list features_OH_onlyMat.csv \
  --fp-list features_FP_onlyMat.csv
"""

import argparse
from pathlib import Path
import pandas as pd
import numpy as np
import sys
import re
import glob

CANDIDATE_NAME_COLS = ["variable","var","name","feature","Feature","column","col","field"]

def detect_name_col(df: pd.DataFrame) -> str:
    # 優先: 既知の列名
    for c in df.columns:
        if str(c).strip() in CANDIDATE_NAME_COLS:
            return c
        if str(c).strip().lower() in [x.lower() for x in CANDIDATE_NAME_COLS]:
            return c
    # 次点: 先頭列
    return df.columns[0]

def read_list(filepath: Path) -> set:
    df = pd.read_csv(filepath)
    col = detect_name_col(df)
    names = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .tolist()
    )
    return set(names)

def normalize_name_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

def filter_file(in_path: Path, keep_set: set, out_dir: Path):
    df = pd.read_csv(in_path)
    name_col = detect_name_col(df)
    before = len(df)

    # 正規化して一致判定
    names_norm = normalize_name_series(df[name_col])
    mask = names_norm.isin(keep_set)
    df2 = df.loc[mask].copy()
    after = len(df2)

    # 保存
    out_dir.mkdir(parents=True, exist_ok=True)
    out_name = in_path.stem + "_onlyMat" + in_path.suffix
    out_path = out_dir / out_name
    df2.to_csv(out_path, index=False, encoding="utf-8-sig")

    return before, after, name_col, out_path

def process_group(assign_root: Path, sub: str, pattern: str, keep_set: set):
    """
    sub: "OH" or "FP"
    pattern: "ClusterAssign_*_OH_*.csv" or "ClusterAssign_*_FP_*.csv"
    keep_set: そのグループで残す変数名セット
    """
    in_dir  = assign_root / sub / "04_assignments"
    out_dir = assign_root / sub / "04_assignments_onlyMat"
    in_dir.mkdir(parents=True, exist_ok=True)
    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(in_dir.glob(pattern))
    logs = []
    for f in files:
        try:
            before, after, name_col, out_path = filter_file(f, keep_set, out_dir)
            logs.append({
                "group": sub,
                "input_file": str(f),
                "output_file": str(out_path),
                "name_col_used": name_col,
                "n_rows_before": before,
                "n_rows_after": after,
                "n_dropped": before - after
            })
        except Exception as e:
            logs.append({
                "group": sub,
                "input_file": str(f),
                "output_file": "",
                "name_col_used": "",
                "n_rows_before": np.nan,
                "n_rows_after": np.nan,
                "n_dropped": np.nan,
                "error": str(e)
            })
    log_df = pd.DataFrame(logs)
    log_path = out_dir / "filter_onlyMat_index.csv"
    log_df.to_csv(log_path, index=False, encoding="utf-8-sig")
    return log_path, log_df

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", required=True, help="Base dir (…/cal_cluster_No)")
    ap.add_argument("--run-root", required=True, help="Run root under base (e.g., 20251026_under_25clusters/STEP3.3_U25_20251027_correct)")
    ap.add_argument("--oh-list", required=True, help="features_OH_onlyMat.csv path (relative to base or absolute)")
    ap.add_argument("--fp-list", required=True, help="features_FP_onlyMat.csv path (relative to base or absolute)")
    args = ap.parse_args()

    base = Path(args.base).resolve()
    run_root = base / args.run_root

    # リストの読み込み（相対なら base から解決）
    oh_list_path = Path(args.oh_list)
    if not oh_list_path.is_absolute():
        oh_list_path = base / oh_list_path
    fp_list_path = Path(args.fp_list)
    if not fp_list_path.is_absolute():
        fp_list_path = base / fp_list_path

    if not oh_list_path.exists():
        sys.exit(f"[ERROR] OH list not found: {oh_list_path}")
    if not fp_list_path.exists():
        sys.exit(f"[ERROR] FP list not found: {fp_list_path}")

    if not run_root.exists():
        sys.exit(f"[ERROR] Run root not found: {run_root}")

    keep_OH = read_list(oh_list_path)
    keep_FP = read_list(fp_list_path)

    # OH 側の一括処理
    log_oh_path, log_oh = process_group(
        assign_root=run_root,
        sub="OH",
        pattern="ClusterAssign_*_OH_*.csv",
        keep_set=keep_OH
    )

    # FP 側の一括処理
    log_fp_path, log_fp = process_group(
        assign_root=run_root,
        sub="FP",
        pattern="ClusterAssign_*_FP_*.csv",
        keep_set=keep_FP
    )

    # 総合ログ
    all_log = pd.concat([log_oh.assign(side="OH"), log_fp.assign(side="FP")], ignore_index=True)
    all_log_path = run_root / "filter_onlyMat_index_ALL.csv"
    all_log.to_csv(all_log_path, index=False, encoding="utf-8-sig")

    print("\n=== Done ===")
    print("OH log:", log_oh_path)
    print("FP log:", log_fp_path)
    print("ALL log:", all_log_path)

if __name__ == "__main__":
    main()
